In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_fcaf_probere1 import ViViTBackbone, ViViTFCAFLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+ViViT + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+2D Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-25 01:49:07,911]::2730006395::DEBUG::=============r2plus1d+ViViT + AST fusion after transformer blocks+2D Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            final_data_set.extend(pickle.load(output_file))
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [5]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [6]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [7]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [8]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)
# Multi-seed setting
SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


1


In [9]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
print(1)


1


In [10]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [11]:
# ============================================================
# ViViT-FCAF Block-wise Linear Probe -- UDIVA / MSE
# ============================================================
# Probe targets:
#   block1 ... block6 : pure ViViT representations
#   final_cross       : representation after final-stage cross-attention
#
# Backbone is frozen. Only probe heads are optimized.
# Evaluation uses direct MSE (lower is better), including O/C/E/A/N.
# ============================================================

PROBE_SEED = 42
probe_lr = 1e-3
probe_weight_decay = 1e-4
probe_epochs = 50
patience = 10
BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_best_multiseed.pth"

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    np.random.seed(worker_seed); random.seed(worker_seed)

set_seed(PROBE_SEED)
g = torch.Generator(); g.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(dataset=train_set_data,batch_size=batchsz,shuffle=True,
    num_workers=num_workerssz,worker_init_fn=seed_worker,generator=g)
val_dataloader = DataLoader(dataset=val_set_data,batch_size=batchsz,shuffle=False,
    num_workers=num_workerssz,worker_init_fn=seed_worker)

backbone = ViViTBackbone(
    2, 14, 14, 2, 2, 2, 5, 96, 6, 8, 384, model=3
).to(device)

ckpt = torch.load(BACKBONE_CHECKPOINT, map_location=device)
state_dict = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
backbone.load_state_dict(state_dict, strict=True)

model = ViViTFCAFLinearProbe(backbone=backbone,dim=96,num_outputs=5).to(device)
backbone_trainable = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
probe_trainable = sum(p.numel() for p in model.probe_heads.parameters() if p.requires_grad)
print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")
print(f"Metric: MSE | Patience: {patience}")
assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.MSELoss().to(device)
optimizer = torch.optim.AdamW(model.probe_heads.parameters(),lr=probe_lr,
                              weight_decay=probe_weight_decay)

probe_names = list(model.probe_heads.keys())
best_mse = {name:float("inf") for name in probe_names}
best_epoch = {name:-1 for name in probe_names}
best_mean_val_mse = float("inf")
no_improve_count = 0

for epoch in range(probe_epochs):
    model.train()
    train_sq_sum = {name:0.0 for name in probe_names}
    train_count = 0

    for audiodata, fullshot, big_five_data in tqdm(
        train_dataloader,desc=f"Probe Train {epoch+1}/{probe_epochs}"
    ):
        fullshot = fullshot.permute(0,4,1,2,3).to(device)
        big_five_data = big_five_data.permute(0,2,1).squeeze().to(device)
        if big_five_data.ndim == 1:
            big_five_data = big_five_data.unsqueeze(0)
        audiodata = audiodata.to(device)

        optimizer.zero_grad()
        outputs = model(fullshot,audiodata)
        losses = {name:criterion(outputs[name],big_five_data) for name in probe_names}
        sum(losses.values()).backward()
        optimizer.step()

        train_count += big_five_data.numel()
        for name in probe_names:
            train_sq_sum[name] += torch.square(
                outputs[name].detach()-big_five_data
            ).sum().item()

    train_mse = {name:train_sq_sum[name]/train_count for name in probe_names}

    model.eval()
    val_sq_sum = {name:0.0 for name in probe_names}
    val_trait_sq_sum = {name:np.zeros(5,dtype=np.float64) for name in probe_names}
    val_count = 0
    val_sample_count = 0

    with torch.no_grad():
        for audiodata, fullshot, big_five_data in tqdm(
            val_dataloader,desc=f"Probe Valid {epoch+1}/{probe_epochs}"
        ):
            fullshot = fullshot.permute(0,4,1,2,3).to(device)
            big_five_data = big_five_data.permute(0,2,1).squeeze().to(device)
            if big_five_data.ndim == 1:
                big_five_data = big_five_data.unsqueeze(0)
            audiodata = audiodata.to(device)
            outputs = model(fullshot,audiodata)

            val_count += big_five_data.numel()
            val_sample_count += big_five_data.shape[0]
            for name in probe_names:
                sq_err = torch.square(outputs[name]-big_five_data)
                val_sq_sum[name] += sq_err.sum().item()
                val_trait_sq_sum[name] += sq_err.sum(dim=0).detach().cpu().numpy()

    val_mse = {name:val_sq_sum[name]/val_count for name in probe_names}
    val_trait_mse = {name:val_trait_sq_sum[name]/val_sample_count for name in probe_names}

    for name in probe_names:
        if val_mse[name] < best_mse[name]:
            best_mse[name] = val_mse[name]
            best_epoch[name] = epoch+1

    mean_val_mse = float(np.mean([val_mse[name] for name in probe_names]))
    if mean_val_mse < best_mean_val_mse:
        best_mean_val_mse = mean_val_mse
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n"+"-"*132)
    print(
        f"Epoch {epoch+1:03d} | Mean Val MSE={mean_val_mse:.6f} | "
        f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
        f"Patience={no_improve_count}/{patience}"
    )
    for name in probe_names:
        o_mse,c_mse,e_mse,a_mse,n_mse = val_trait_mse[name]
        print(
            f"{name:<12} | Train MSE {train_mse[name]:.6f} | "
            f"Val MSE {val_mse[name]:.6f} | Best MSE {best_mse[name]:.6f} | "
            f"O MSE {o_mse:.6f} | C MSE {c_mse:.6f} | E MSE {e_mse:.6f} | "
            f"A MSE {a_mse:.6f} | N MSE {n_mse:.6f}"
        )

    if no_improve_count >= patience:
        print(
            f"\nEarly stopping at epoch {epoch+1} | "
            f"Best Mean Val MSE={best_mean_val_mse:.6f} | "
            f"Patience={no_improve_count}/{patience}"
        )
        break
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n"+"="*72)
print("FINAL ViViT-FCAF LINEAR PROBE RESULTS -- UDIVA / MSE")
print("="*72)
print(f"{'Representation':<18}{'Best Epoch':>12}{'Best MSE':>16}")
print("-"*72)
for name in probe_names:
    print(f"{name:<18}{best_epoch[name]:>12d}{best_mse[name]:>16.6f}")
print("="*72)

del model, backbone, optimizer, train_dataloader, val_dataloader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
Backbone trainable parameters: 0
Probe trainable parameters:    4739
Metric: MSE | Patience: 10


Probe Valid 1/50: 100%|██████████| 191/191 [00:31<00:00,  6.02it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 001 | Mean Val MSE=1.233767 | Best Mean Val MSE=1.233767 | Patience=0/10
block1       | Train MSE 0.113936 | Val MSE 1.223396 | Best MSE 1.223396 | O MSE 1.043359 | C MSE 1.282180 | E MSE 1.081441 | A MSE 1.423023 | N MSE 1.286974
block2       | Train MSE 0.106949 | Val MSE 1.223284 | Best MSE 1.223284 | O MSE 1.053677 | C MSE 1.292757 | E MSE 1.085418 | A MSE 1.404387 | N MSE 1.280179
block3       | Train MSE 0.085310 | Val MSE 1.236830 | Best MSE 1.236830 | O MSE 1.085324 | C MSE 1.290278 | E MSE 1.109797 | A MSE 1.420322 | N MSE 1.278427
block4       | Train MSE 0.114146 | Val MSE 1.227080 | Best MSE 1.227080 | O MSE 1.023985 | C MSE 1.308962 | E MSE 1.125627 | A MSE 1.426186 | N MSE 1.250638
block5       | Train MSE 0.092550 | Val MSE 1.211542 | Best MSE 1.211542 | O MSE 1.043072 | C MSE 1.312189 | E MSE 1.099171 | A MSE 1.409842 | N MSE 1.1934

Probe Valid 2/50: 100%|██████████| 191/191 [00:33<00:00,  5.77it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 002 | Mean Val MSE=1.221122 | Best Mean Val MSE=1.221122 | Patience=0/10
block1       | Train MSE 0.042404 | Val MSE 1.208953 | Best MSE 1.208953 | O MSE 1.071442 | C MSE 1.282086 | E MSE 1.107028 | A MSE 1.381268 | N MSE 1.202941
block2       | Train MSE 0.037242 | Val MSE 1.205827 | Best MSE 1.205827 | O MSE 1.095257 | C MSE 1.292839 | E MSE 1.080881 | A MSE 1.371939 | N MSE 1.188222
block3       | Train MSE 0.034144 | Val MSE 1.219138 | Best MSE 1.219138 | O MSE 1.117152 | C MSE 1.282387 | E MSE 1.113194 | A MSE 1.385159 | N MSE 1.197800
block4       | Train MSE 0.030201 | Val MSE 1.212819 | Best MSE 1.212819 | O MSE 1.058796 | C MSE 1.297353 | E MSE 1.142602 | A MSE 1.389389 | N MSE 1.175954
block5       | Train MSE 0.026529 | Val MSE 1.202006 | Best MSE 1.202006 | O MSE 1.085021 | C MSE 1.284671 | E MSE 1.114068 | A MSE 1.383376 | N MSE 1.1428

...


------------------------------------------------------------------------------------------------------------------------------------
Epoch 025 | Mean Val MSE=1.220613 | Best Mean Val MSE=1.201503 | Patience=8/10
block1       | Train MSE 0.025045 | Val MSE 1.206132 | Best MSE 1.184892 | O MSE 0.956727 | C MSE 1.284971 | E MSE 1.152486 | A MSE 1.401009 | N MSE 1.235466
block2       | Train MSE 0.021919 | Val MSE 1.206166 | Best MSE 1.181426 | O MSE 0.971960 | C MSE 1.269142 | E MSE 1.192593 | A MSE 1.387961 | N MSE 1.209172
block3       | Train MSE 0.019938 | Val MSE 1.219455 | Best MSE 1.192253 | O MSE 0.981356 | C MSE 1.257858 | E MSE 1.251048 | A MSE 1.396072 | N MSE 1.210942
block4       | Train MSE 0.017586 | Val MSE 1.206815 | Best MSE 1.184461 | O MSE 0.980192 | C MSE 1.261958 | E MSE 1.213931 | A MSE 1.402815 | N MSE 1.175178
block5       | Train MSE 0.015888 | Val MSE 1.211086 | Best MSE 1.184048 | O MSE 1.004798 | C MSE 1.270318 | E MSE 1.223234 | A MSE 1.394071 | N MSE 1.1630

Probe Valid 26/50: 100%|██████████| 191/191 [00:20<00:00,  9.12it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 026 | Mean Val MSE=1.205968 | Best Mean Val MSE=1.201503 | Patience=9/10
block1       | Train MSE 0.024451 | Val MSE 1.193356 | Best MSE 1.184892 | O MSE 0.971575 | C MSE 1.257312 | E MSE 1.104475 | A MSE 1.398228 | N MSE 1.235191
block2       | Train MSE 0.021504 | Val MSE 1.189496 | Best MSE 1.181426 | O MSE 0.991652 | C MSE 1.233153 | E MSE 1.131524 | A MSE 1.377032 | N MSE 1.214119
block3       | Train MSE 0.019481 | Val MSE 1.199522 | Best MSE 1.192253 | O MSE 1.000420 | C MSE 1.228609 | E MSE 1.179492 | A MSE 1.382368 | N MSE 1.206719
block4       | Train MSE 0.017161 | Val MSE 1.186523 | Best MSE 1.184461 | O MSE 1.002165 | C MSE 1.229305 | E MSE 1.136985 | A MSE 1.389141 | N MSE 1.175017
block5       | Train MSE 0.015539 | Val MSE 1.195124 | Best MSE 1.184048 | O MSE 1.042216 | C MSE 1.229275 | E MSE 1.146037 | A MSE 1.380851 | N MSE 1.1772

Probe Valid 27/50: 100%|██████████| 191/191 [00:20<00:00,  9.14it/s]



------------------------------------------------------------------------------------------------------------------------------------
Epoch 027 | Mean Val MSE=1.217631 | Best Mean Val MSE=1.201503 | Patience=10/10
block1       | Train MSE 0.024545 | Val MSE 1.206448 | Best MSE 1.184892 | O MSE 0.959541 | C MSE 1.289200 | E MSE 1.130136 | A MSE 1.401251 | N MSE 1.252109
block2       | Train MSE 0.021526 | Val MSE 1.203880 | Best MSE 1.181426 | O MSE 0.976021 | C MSE 1.272125 | E MSE 1.157176 | A MSE 1.385190 | N MSE 1.228888
block3       | Train MSE 0.019516 | Val MSE 1.214923 | Best MSE 1.192253 | O MSE 0.982680 | C MSE 1.267943 | E MSE 1.210272 | A MSE 1.395238 | N MSE 1.218482
block4       | Train MSE 0.017209 | Val MSE 1.202509 | Best MSE 1.184461 | O MSE 0.976549 | C MSE 1.269788 | E MSE 1.179970 | A MSE 1.401857 | N MSE 1.184383
block5       | Train MSE 0.015653 | Val MSE 1.203181 | Best MSE 1.184048 | O MSE 1.002746 | C MSE 1.255169 | E MSE 1.194990 | A MSE 1.381991 | N MSE 1.181